# E0 - Environment log

Camera sensor temperature, enclosure temperature, humidity and light (SHT45 + TSL2591 on the UC2
CAN bus, through `I2CSensorController`) of one or more ImSwitch servers over time, into one CSV.

Every experiment notebook (E1-E4) already logs its own camera and enclosure into its run folder and
can add the other server (`OTHER_SERVERS`). Use this one when no experiment runs, e.g. to check
that the enclosure stays dark overnight, or to log both rigs side by side.

The enclosure sensors sit on a CAN slave: if the bus power is switched off during an exposure (as
the long-term dark-cycle notebook does), they read nothing for that time.

## 1 Settings

In [ ]:
from pathlib import Path
import biophoton as bp

RUN_NAME = "enclosure"
PERIOD_S = 30
SERVERS = [   # url None = $IMSWITCH_API_URL; detector None = first detector of that server
    {"label": "toupcam", "url": None, "detector": None},
    # {"label": "andor", "url": "http://192.168.1.20:8001", "detector": "Andor"},
]
DATA_ROOT = Path("data")

## 2 Start logging

Runs in background threads, one per server.

In [ ]:
try:
    env.stop()          # re-running replaces the old logger
except NameError:
    pass

run = bp.Run("E0_environment", RUN_NAME, DATA_ROOT)
sources = []
for s in SERVERS:
    c = bp.connect(s["url"])
    det = s.get("detector") or c.settingsManager.getDetectorNames()[0]
    sources.append({"label": s["label"], "client": c, "detector": det})
    print(f"{s['label']}: {c.base_uri} {det} camera {bp.camera_temperature(c, det)} "
          f"enclosure {bp.enclosure_values(c)}")
run.update_config(kind="environment", experiment="E0", title="E0 - environment log",
                  description="Camera and enclosure temperature, humidity and light over time.",
                  servers=[{"label": s["label"], "url": s["client"].base_uri, "detector": s["detector"]}
                           for s in sources], period_s=PERIOD_S)
env = bp.EnvLogger(run.dir / "environment.csv", sources, PERIOD_S).start()

## 3 Live view

Interrupt (■) to stop the display; logging continues.

In [ ]:
bp.dashboard(run.dir, refresh_s=30)

## 4 Stop and report

In [ ]:
env.stop()
RUN_DIR = run.dir           # or an older run folder
summary = bp.redraw(RUN_DIR, show=True)
summary